# crypto-vol-surface: demo

From a Deribit snapshot to arbitrage-free SVI / SSVI surfaces, smile analytics and Black-76 repricing.

The notebook uses the latest local snapshot in `data/snapshots/` if one exists (create one with
`python scripts/refresh_snapshots.py`). Otherwise it falls back to the committed **synthetic**
fixture, which has a known SSVI ground truth. Set `CRYPTOVOL_SYNTHETIC=1` to force the fixture.

Deribit market data, and anything derived from it, is personal-use only and is never committed (see `DATA.md`). The outputs saved
in this notebook were produced on the **synthetic fixture**, so they contain no Deribit data.

In [1]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cryptovol.report import analyse
from cryptovol.snapshots import latest_snapshot_dir, load_snapshot

pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
synthetic = ROOT / "tests" / "fixtures" / "synthetic"
if os.environ.get("CRYPTOVOL_SYNTHETIC") == "1":
    path = latest_snapshot_dir(synthetic)
else:
    path = latest_snapshot_dir(ROOT / "data" / "snapshots") or latest_snapshot_dir(synthetic)
snap = load_snapshot(path)
print("snapshot:", path.name, "|", snap.meta.get("source"))

snapshot: 2026-01-02T080000Z | synthetic (cryptovol.synthetic, known SSVI ground truth)


## 1. Clean, fit and analyse each currency

In [2]:
reports = {ccy: analyse(snap, ccy) for ccy in snap.currencies()}
for ccy, rep in reports.items():
    flags = rep.quotes["flag"].value_counts().to_dict()
    n_kept = int(rep.quotes["keep"].sum())
    print(f"{ccy}: {n_kept} of {len(rep.quotes)} OTM quotes kept; flags = {flags}")

BTC: 937 of 1072 OTM quotes kept; flags = {'ok': 937, 'far_wing': 121, 'wide_spread': 14}
ETH: 2016 of 2098 OTM quotes kept; flags = {'ok': 2016, 'far_wing': 76, 'wide_spread': 6}


### Forwards: futures vs put-call parity
`parity_minus_future_bp` is the gap between the put-call-parity forward and the futures mid (basis points); `basis_ann` is the annualised log-basis $`\ln(F/S)/T`$.

In [3]:
fwd = reports["BTC"].forwards
fwd.assign(
    days=365 * fwd["T"],
    parity_minus_future_bp=1e4 * (fwd["parity_forward"] / fwd["future_mid"] - 1),
)[["days", "forward_source", "parity_minus_future_bp", "basis_ann"]]

,days,forward_source,parity_minus_future_bp,basis_ann
0,7.0,future,0.0082,0.06
1,14.0,future,-0.0588,0.06
2,28.0,future,-0.0217,0.06
3,56.0,future,-0.1347,0.06
4,91.0,future,-0.0335,0.06
5,182.0,future,-0.1566,0.06
6,364.0,future,0.0606,0.06


## 2. SVI per expiry vs SSVI surface
RMSE against mid IV (vol points) and the share of quotes repriced inside the bid-ask.

In [4]:
reports["BTC"].comparison.drop(columns="F")

,expiry,days,n,svi_rmse_volpts,ssvi_rmse_volpts,svi_inside_share,ssvi_inside_share,svi_butterfly_ok
0,2026-01-09 08:00:00+00:00,7.0,32,0.1559,0.1641,1.0,1.0,True
1,2026-01-16 08:00:00+00:00,14.0,46,0.0815,0.0979,1.0,1.0,True
2,2026-01-30 08:00:00+00:00,28.0,65,0.0490,0.0528,1.0,1.0,True
3,2026-02-27 08:00:00+00:00,56.0,96,0.0490,0.0500,1.0,1.0,True
4,2026-04-03 08:00:00+00:00,91.0,127,0.0374,0.0378,1.0,1.0,True
5,2026-07-03 08:00:00+00:00,182.0,203,0.0250,0.0256,1.0,1.0,True
6,2027-01-01 08:00:00+00:00,364.0,368,0.0196,0.0197,1.0,1.0,True


In [5]:
pd.concat({c: r.ssvi_params() for c, r in reports.items()})

,,rho,eta,gamma,rmse_volpts,theta_monotone,max_theta_phi_1p_rho,max_theta_phi2_1p_rho,min_g,min_calendar_gap,arbitrage_free
BTC,0,-0.2501,1.0012,0.4494,0.0483,True,0.5377,0.8902,0.2756,0.0045,True
ETH,0,-0.1500,1.1999,0.4001,0.0447,True,0.6929,0.9633,0.2860,0.0080,True


## 3. ATM term structure and 25-delta RR / BF (forward vs premium-adjusted delta)

In [6]:
rows = []
for ccy, rep in reports.items():
    d = rep.rr_bf[rep.rr_bf["model"] == "svi"]
    piv = d.pivot_table(index="days", columns="convention", values=["rr", "bf"]) * 100
    piv.columns = [f"{a}_{b}" for a, b in piv.columns]
    rows.append(piv.assign(currency=ccy, atm=100 * rep.atm.set_index("days")["svi"]))
pd.concat(rows).round(2)

,bf_forward,bf_premium_adjusted,rr_forward,rr_premium_adjusted,currency,atm
days,,,,,,
7.0,1.48,1.64,-6.10,-6.44,BTC,50.34
14.0,1.47,1.71,-6.16,-6.66,BTC,50.52
28.0,1.48,1.83,-6.12,-6.87,BTC,50.70
56.0,1.44,1.93,-5.93,-7.06,BTC,50.98
91.0,1.35,1.96,-5.68,-7.15,BTC,51.25
182.0,1.13,1.94,-5.07,-7.13,BTC,51.77
364.0,0.76,1.79,-4.16,-6.90,BTC,52.50
7.0,1.83,1.98,-4.33,-4.85,ETH,65.44
14.0,2.03,2.26,-4.33,-5.17,ETH,65.63


In [7]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for ccy, rep in reports.items():
    d = rep.rr_bf[(rep.rr_bf["model"] == "svi") & (rep.rr_bf["convention"] == "forward")]
    ax.plot(d["days"], 100 * d["rr"], marker="o", label=f"{ccy} 25d RR")
ax.axhline(0, lw=0.8, color="grey")
ax.set_xscale("log")
ax.set_xlabel("days")
ax.set_ylabel("vol points")
ax.legend()
ax.set_title("25-delta risk reversal (forward delta)", loc="left");

## 4. Fitted IV vs Deribit mark IV

In [8]:
pd.DataFrame(
    {
        ccy: {
            "median |SVI - mark|, vol pts": rep.errors_svi["err_vs_mark_volpts"].abs().median(),
            "median |SSVI - mark|, vol pts": rep.errors_ssvi["err_vs_mark_volpts"].abs().median(),
            "median |mid - mark|, vol pts": rep.errors_svi["mid_minus_mark_volpts"].abs().median(),
        }
        for ccy, rep in reports.items()
    }
).round(3)

,BTC,ETH
"median |SVI - mark|, vol pts",0.069,0.070
"median |SSVI - mark|, vol pts",0.069,0.070
"median |mid - mark|, vol pts",0.074,0.073


## 5. Pricing and Greeks off the fitted surface
Sticky-strike Black-76 Greeks in USD per unit of underlying. `delta_pa` is the premium-adjusted delta of the inverse option.

In [9]:
s = reports["BTC"].fits.svi
F1m = float(s.forward(30 / 365))
K = np.round(F1m * np.array([0.85, 0.95, 1.0, 1.05, 1.15]), -3)
s.greeks(K, 30 / 365, F1m <= K)

,K,T,is_call,F,iv,price_usd,delta,delta_pa,gamma,vega_1pct,theta_1d
0,77000.0,0.0822,False,90444.9318,0.5922,1299.4296,-0.1509,-0.1652,1.5241e-05,60.6863,-59.8990
1,86000.0,0.0822,False,90444.9318,0.5290,3400.9564,-0.3416,-0.3792,2.6762e-05,95.1784,-83.9105
2,90000.0,0.0822,False,90444.9318,0.5091,5029.5327,-0.4575,-0.5131,3.0048e-05,102.8568,-87.2782
3,95000.0,0.0822,True,90444.9318,0.4946,3273.9991,0.3914,0.3552,2.9949e-05,99.5887,-82.0904
4,104000.0,0.0822,True,90444.9318,0.4957,1185.0628,0.1810,0.1679,2.0485e-05,68.2775,-56.4122


## 6. Arbitrage checks

In [10]:
for ccy, rep in reports.items():
    print(
        ccy,
        "| SVI slices butterfly-free:",
        bool(rep.comparison["svi_butterfly_ok"].all()),
        "| SVI calendar crossings (k in [-1.5, 1.5]):",
        int((rep.svi_calendar_gaps < 0).sum()),
        "| SSVI arbitrage-free:",
        bool(rep.ssvi_params()["arbitrage_free"].iloc[0]),
    )

BTC | SVI slices butterfly-free: True | SVI calendar crossings (k in [-1.5, 1.5]): 0 | SSVI arbitrage-free: True
ETH | SVI slices butterfly-free: True | SVI calendar crossings (k in [-1.5, 1.5]): 0 | SSVI arbitrage-free: True
